# Holo-GNN: V5.0 Production Run (Full MegaScale Dataset)

**Architecture upgrades over V4.0:**
- `GATConv` → **`GATv2Conv`** (Brody et al. 2022) — dynamic attention; removes the rank-1 static-attention bottleneck of GATv1
- **Residual skip connection** + `LayerNorm` around graph layers — prevents over-smoothing across 780k samples
- **True CAI** from E. coli K-12 codon usage table (Sharp & Li 1987)
- **True Henderson-Hasselbalch charge** model at pH 7.4 with published pKa values
- **GC-skew / stacking mRNA fold proxy** (no ViennaRNA dependency)

This notebook executes the full V5.0 production pipeline:
- **Phase 1 — Training:** 5-epoch Siamese loop on the **full** 780k-mutation MegaScale dataset, `CosineAnnealingWarmRestarts` scheduler, epoch-level checkpointing every epoch
- **Phase 2 — Benchmarking:** RMSE, MAE, Pearson *r*, mean antisymmetry violation on held-out test set
- **Phase 3 — Visualisation:** Loss curves, scatter plot, violation histogram → `holognn_v5_metrics.png`

---
> **Vertex AI target:** NVIDIA L4 GPU (24 GB VRAM) · 8 vCPU workers  
> **Run:** `Runtime → Run all`  
> **Preemption-safe:** epoch checkpoint saved after every epoch regardless of best val loss

In [ ]:
# Cell 2: Install required domain-specific libraries for Vertex AI environment
!pip install "numpy<2" "transformers==4.33.0" biopython torch_geometric

# Verify GATv2Conv is available (requires torch_geometric >= 2.0)
try:
    from torch_geometric.nn import GATv2Conv
    print(f"✅ GATv2Conv available")
except ImportError:
    print("⚠️  GATv2Conv not found in this torch_geometric version — will fall back to GATConv")

In [ ]:
# Cell 3: Dataset Initialisation — Full MegaScale, no MAX_SAMPLES limit
# ═══════════════════════════════════════════════════════════════════════

import os, math, time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.notebook import tqdm
from transformers import EsmTokenizer, EsmModel
from Bio.Seq import Seq
from scipy.stats import pearsonr
import matplotlib
matplotlib.use("Agg")   # headless-safe
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ── PyG (GATv2Conv preferred; GATConv fallback) ───────────────────────────────
try:
    from torch_geometric.nn import GATv2Conv
    _PYGEO = True
    _GAT_CLS = GATv2Conv
    print("✅ GATv2Conv — V5.0 dynamic attention ENABLED")
except ImportError:
    try:
        from torch_geometric.nn import GATConv as GATv2Conv
        _PYGEO = True
        _GAT_CLS = GATv2Conv
        print("⚠️  Falling back to GATConv (V3.0 static attention)")
    except ImportError:
        GATv2Conv = None
        _PYGEO = False
        _GAT_CLS = None
        print("⚠️  torch_geometric missing — ESM-2 mean-pool fallback active")

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   Device: {device}")

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH       = (
    "data/mega_scale_cdna/Processed_K50_dG_datasets/"
    "Processed_K50_dG_datasets/Tsuboyama2023_Dataset1_20230416.csv"
)
CHECKPOINT_BEST  = "holognn_v5_best.pth"
CHECKPOINT_EPOCH = "holognn_v5_epoch_{epoch}.pth"   # preemption-safe per-epoch saves

# ── Hyperparameters ───────────────────────────────────────────────────────────
# No MAX_SAMPLES — load the full 780k-mutation dataset
MAX_SEQ_LEN   = 100
BATCH_SIZE    = 64         # L4 24 GB VRAM
NUM_WORKERS   = 8          # matches vCPU count
EPOCHS        = 5
LEARNING_RATE = 1e-4
ALPHA_ASYM    = 1.0        # AntisymmetricLoss antisymmetry weight
VAL_SPLIT     = 0.10
TEST_SPLIT    = 0.10
# CosineAnnealingWarmRestarts: restart every T_0 epochs
COSINE_T0     = 2

# ════════════════════════════════════════════════════
# V5.0 Mechanistic Feature Tables
# ════════════════════════════════════════════════════

# True CAI weights — E. coli K-12 high-expression reference (Sharp & Li 1987)
_CODON_CAI_WEIGHT = {
    "TTT": 0.296, "TTC": 1.000,
    "TTA": 0.049, "TTG": 0.100, "CTT": 0.100, "CTC": 0.073, "CTA": 0.039, "CTG": 1.000,
    "ATT": 0.731, "ATC": 1.000, "ATA": 0.107, "ATG": 1.000,
    "GTT": 0.726, "GTC": 0.354, "GTA": 0.378, "GTG": 1.000,
    "TCT": 1.000, "TCC": 0.744, "TCA": 0.298, "TCG": 0.260, "AGT": 0.209, "AGC": 0.828,
    "CCT": 0.516, "CCC": 0.195, "CCA": 0.271, "CCG": 1.000,
    "ACT": 0.965, "ACC": 1.000, "ACA": 0.308, "ACG": 0.424,
    "GCT": 1.000, "GCC": 0.556, "GCA": 0.469, "GCG": 0.636,
    "TAT": 0.326, "TAC": 1.000, "CAT": 0.424, "CAC": 1.000,
    "CAA": 0.124, "CAG": 1.000, "AAT": 0.366, "AAC": 1.000,
    "AAA": 1.000, "AAG": 0.248, "GAT": 0.776, "GAC": 1.000,
    "GAA": 1.000, "GAG": 0.356, "TGT": 0.500, "TGC": 1.000, "TGG": 1.000,
    "CGT": 1.000, "CGC": 0.758, "CGA": 0.111, "CGG": 0.111, "AGA": 0.070, "AGG": 0.070,
    "GGT": 1.000, "GGC": 0.724, "GGA": 0.145, "GGG": 0.181,
}

# Henderson-Hasselbalch pKa values (Thurlkill et al. 2006) and charge sign
_PKA = {
    'D': (3.67, -1.0), 'E': (4.25, -1.0), 'H': (6.54, +1.0),
    'C': (8.18, -1.0), 'Y': (10.00, -1.0), 'K': (10.53, +1.0), 'R': (12.00, +1.0),
}
_PH = 7.4
_CODON_COMPOSITION: dict = {}


def _hh_charge(aa: str) -> float:
    """Fractional Henderson-Hasselbalch charge at pH 7.4."""
    if aa not in _PKA: return 0.0
    pka, q_sign = _PKA[aa]
    if q_sign < 0:
        return -1.0 / (1.0 + 10.0 ** (pka - _PH))
    return +1.0 / (1.0 + 10.0 ** (_PH - pka))


def _nt_counts(codon: str):
    if codon not in _CODON_COMPOSITION:
        _CODON_COMPOSITION[codon] = (
            codon.count('A'), codon.count('T'),
            codon.count('G'), codon.count('C')
        )
    return _CODON_COMPOSITION[codon]


def _mechanistic_features(protein_seq: str, dna_seq: str, max_length: int) -> torch.Tensor:
    """
    V5.0: True per-residue mechanistic features → (max_length, 3).
      ch0 — mRNA_fold : GC-skew / stacking energy proxy (codon window)
      ch1 — CAI       : E. coli K-12 reference codon weight [0, 1]
      ch2 — Charge    : H-H fractional charge at pH 7.4, scaled [0, 1]
    """
    n_codons = len(dna_seq) // 3
    L        = min(len(protein_seq), max_length)
    feat     = torch.zeros(max_length, 3, dtype=torch.float32)
    half_c   = 1   # codon window ±1 (3 codons total)
    half_ch  = 3   # charge window ±3 (7 residues total)

    for i in range(L):
        # ch0 — mRNA fold (GC-skew + GC-fraction over ±1 codon window)
        A = T = G = C = 0
        for ci in range(max(0, i - half_c), min(n_codons, i + half_c + 1)):
            codon = dna_seq[ci*3: ci*3+3].upper().replace('U', 'T')
            if len(codon) == 3:
                a, t, g, c = _nt_counts(codon)
                A+=a; T+=t; G+=g; C+=c
        total    = A + T + G + C
        gc_frac  = (G + C) / total if total else 0.0
        gc_skew  = (G - C) / (G + C) if (G + C) > 0 else 0.0
        mrna     = 0.5 * gc_frac + 0.5 * (1.0 - abs(gc_skew))

        # ch1 — CAI weight for this codon
        codon_i = dna_seq[i*3: i*3+3].upper() if i < n_codons else ""
        cai     = _CODON_CAI_WEIGHT.get(codon_i, 0.5)

        # ch2 — H-H charge (scaled to [0,1] from [-7,+7])
        raw_ch  = sum(
            _hh_charge(protein_seq[j].upper())
            for j in range(max(0, i - half_ch), min(len(protein_seq), i + half_ch + 1))
        )
        charge  = max(0.0, min(1.0, (raw_ch + 7.0) / 14.0))

        feat[i] = torch.tensor([mrna, cai, charge])
    return feat


# ════════════════════════════════════════════════════════════════════
# V5.0 MegaScaleDataset — FULL dataset, no MAX_SAMPLES limit
# ════════════════════════════════════════════════════════════════════
class MegaScaleDataset(Dataset):
    """
    Full MegaScale cDNA dataset (~780k mutations).
    No sample cap — loads the entire dataframe.
    Returns: input_ids, attention_mask, label, mechanistic_features
    """
    def __init__(self, csv_path: str, max_length: int = MAX_SEQ_LEN):
        self.tok        = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
        self.max_length = max_length
        print(f"Loading FULL MegaScale dataset from {csv_path} ...")
        df       = pd.read_csv(csv_path)
        self.df  = df.dropna(subset=["dna_seq", "deltaG"]).reset_index(drop=True)
        print(f"  {len(self.df):,} valid samples — no MAX_SAMPLES cap.")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        dna_seq     = str(row["dna_seq"])
        protein_seq = str(Seq(dna_seq).translate(to_stop=True))
        label       = float(row["deltaG"])
        enc = self.tok(
            protein_seq, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids":            enc["input_ids"].squeeze(0),
            "attention_mask":       enc["attention_mask"].squeeze(0),
            "label":                torch.tensor(label, dtype=torch.float),
            "mechanistic_features": _mechanistic_features(protein_seq, dna_seq, self.max_length),
        }


# ── Build dataset & splits (full data, no cap) ────────────────────────────────
full_ds  = MegaScaleDataset(DATA_PATH)
n        = len(full_ds)   # ALL samples — no Subset truncation

n_test   = int(TEST_SPLIT * n)
n_val    = int(VAL_SPLIT  * n)
n_train  = n - n_val - n_test
train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test])

_lkw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
             pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, shuffle=True,  **_lkw)
val_loader   = DataLoader(val_ds,   shuffle=False, **_lkw)
test_loader  = DataLoader(test_ds,  shuffle=False, **_lkw)

print(f"\n✅ Full-scale DataLoaders ready.")
print(f"   Train : {len(train_ds):,} samples | {len(train_loader):,} batches")
print(f"   Val   : {len(val_ds):,}  samples | {len(val_loader):,}  batches")
print(f"   Test  : {len(test_ds):,}  samples | {len(test_loader):,}  batches")

In [ ]:
# Cell 4: V5.0 Phase Generator — Production Training Loop
# ═══════════════════════════════════════════════════════════════════════════
#  Phase 1 : Siamese training (CosineAnnealingWarmRestarts + epoch checkpoints)
#  Phase 2 : Test-set benchmarking  (RMSE, MAE, Pearson r, antisymmetry)
#  Phase 3 : Matplotlib visualisation → holognn_v5_metrics.png
# ═══════════════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════
# Architecture — V5.0 inlined (no local src/ package)
# ════════════════════════════════════════════════════════

ESM2_HIDDEN_DIM  = 320
MECH_FEATURE_DIM = 3
GAT_IN_CHANNELS  = ESM2_HIDDEN_DIM + MECH_FEATURE_DIM   # 323


def build_attention_graph(avg_attention: torch.Tensor, threshold: float = 0.05) -> torch.Tensor:
    adj        = avg_attention + avg_attention.t()
    rows, cols = torch.where(adj > threshold)
    mask       = rows != cols
    return torch.stack([rows[mask], cols[mask]], dim=0)


# ── V5.0 Backbone (GATv2Conv + Residual + LayerNorm) ─────────────────────────
class HoloGNNBackbone(nn.Module):
    """
    V5.0: ESM-2 → Mechanistic Injection → Attention Graph
         → GATv2Conv × 2 → Residual Skip + LayerNorm → Mean Pool
    """
    def __init__(self, output_dim: int = 320):
        super().__init__()
        self.esm = EsmModel.from_pretrained(
            "facebook/esm2_t6_8M_UR50D", output_attentions=True
        )
        if _PYGEO:
            self.gat1 = _GAT_CLS(GAT_IN_CHANNELS, GAT_IN_CHANNELS, heads=4, concat=False)
            self.gat2 = _GAT_CLS(GAT_IN_CHANNELS, output_dim,      heads=4, concat=False)
        # Residual projection: ESM-2 (320) → output_dim
        self.residual_proj = (
            nn.Linear(ESM2_HIDDEN_DIM, output_dim, bias=False)
            if ESM2_HIDDEN_DIM != output_dim else nn.Identity()
        )
        self.layer_norm = nn.LayerNorm(output_dim)
        self.relu       = nn.ReLU()

    def forward(self, input_ids, attention_mask, mechanistic_features, edge_index=None):
        B = input_ids.size(0)

        # Step 1 — ESM-2
        esm_out    = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        esm_nodes  = esm_out.last_hidden_state              # (B, L, 320)
        L          = esm_nodes.size(1)

        # Step 2 — Mechanistic injection
        node_emb   = torch.cat([esm_nodes, mechanistic_features], dim=-1)  # (B, L, 323)

        # Step 3 — Attention graph
        if edge_index is None and _PYGEO:
            last_attn  = esm_out.attentions[-1]             # (B, heads, L, L)
            batch_avg  = torch.mean(torch.mean(last_attn, dim=1), dim=0)   # (L, L)
            edge_index = build_attention_graph(batch_avg).to(input_ids.device)

        # Step 4 — GATv2Conv message passing
        x = node_emb.view(-1, node_emb.size(-1))            # (B*L, 323)
        if edge_index is not None and _PYGEO:
            x = self.relu(self.gat1(x, edge_index))         # (B*L, 323)
            x = self.gat2(x, edge_index)                    # (B*L, output_dim)

        # Step 5 — Residual skip connection
        #   Add projected raw ESM-2 embeddings back to GATv2 output.
        #   Prevents over-smoothing; preserves sequence identity signal.
        esm_flat  = esm_nodes.view(-1, ESM2_HIDDEN_DIM)     # (B*L, 320)
        residual  = self.residual_proj(esm_flat)             # (B*L, output_dim)
        if x.size(-1) != residual.size(-1):
            x = x[..., :residual.size(-1)]                  # trim fallback path
        x = self.layer_norm(x + residual)                   # (B*L, output_dim)

        # Step 6 — Mean pool
        x_r       = x.view(B, L, -1)
        graph_emb = torch.mean(x_r, dim=1)                  # (B, output_dim)
        return x_r, graph_emb


# ── Heads ─────────────────────────────────────────────────────────────────────
class ProteomicsHead(nn.Module):
    def __init__(self, d=320): super().__init__(); self.r = nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Linear(256,1))
    def forward(self,z): return self.r(z)

class SiameseStabilityHead(nn.Module):
    def __init__(self, d=320): super().__init__(); self.mlp = nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Linear(256,1))
    def forward(self,z_wt,z_mt): return self.mlp(z_mt - z_wt)

class EnsembleIDRHead(nn.Module):
    def __init__(self, d=320):
        super().__init__()
        self.mu=nn.Linear(d,1); self.sigma=nn.Linear(d,1); self.sp=nn.Softplus()
    def forward(self,z): return self.mu(z), self.sp(self.sigma(z))


# ── V4.0/V5.0 Full Model ─────────────────────────────────────────────────────
class _DataBatch:
    __slots__ = ("input_ids", "mask", "mechanistic_features", "edge_index")

class HoloGNN(nn.Module):
    """V5.0 HoloGNN — Siamese + GATv2Conv + Residual."""
    def __init__(self):
        super().__init__()
        self.backbone        = HoloGNNBackbone(output_dim=320)
        self.proteomics_head = ProteomicsHead(320)
        self.siamese_head    = SiameseStabilityHead(320)
        self.idr_head        = EnsembleIDRHead(320)

    def _encode(self, data):
        _, z = self.backbone(
            data.input_ids, data.mask, data.mechanistic_features, data.edge_index
        )
        return z

    def forward(self, data, task="proteomics"):
        if task == "proteomics":  return self.proteomics_head(self._encode(data))
        if task == "idr":
            data_wt, data_mt = data
            z_wt, z_mt = self._encode(data_wt), self._encode(data_mt)
            return self.siamese_head(z_wt, z_mt), self.siamese_head(z_mt, z_wt)
        return self._encode(data)


# ── AntisymmetricLoss ─────────────────────────────────────────────────────────
class AntisymmetricLoss(nn.Module):
    """L = alpha*(fwd+rev)^2 + (fwd-exp)^2"""
    def __init__(self, alpha=1.0): super().__init__(); self.alpha=alpha
    def forward(self, dG_fwd, dG_rev, dG_exp):
        fwd=dG_fwd.squeeze(-1); rev=dG_rev.squeeze(-1)
        asym = (fwd+rev)**2; fidel = (fwd-dG_exp)**2
        return (torch.mean(self.alpha*asym+fidel),
                {"antisymmetry": torch.mean(asym).item(), "fidelity": torch.mean(fidel).item()})


# ── Siamese batch builder ─────────────────────────────────────────────────────
def make_siamese_pair(batch, dev):
    half = batch["input_ids"].size(0) // 2
    if half == 0: raise ValueError("Batch too small for Siamese split")
    def _mk(sl):
        d = _DataBatch()
        d.input_ids            = batch["input_ids"][sl].to(dev, non_blocking=True)
        d.mask                 = batch["attention_mask"][sl].to(dev, non_blocking=True)
        d.mechanistic_features = batch["mechanistic_features"][sl].to(dev, non_blocking=True)
        d.edge_index           = None
        return d
    return _mk(slice(None, half)), _mk(slice(half, half*2)), batch["label"][:half].to(dev, non_blocking=True)


# ════════════════════════════════════════════════════════════════════
# Model + Optimiser + CosineAnnealingWarmRestarts Scheduler
# ════════════════════════════════════════════════════════════════════
print(f"Initialising HoloGNN V5.0 on {device} ...")
model     = HoloGNN().to(device)
criterion = AntisymmetricLoss(alpha=ALPHA_ASYM)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# CosineAnnealingWarmRestarts:
#   T_0 = number of steps in the first restart cycle.
#   At 780k samples × 80 % train / batch 64 ≈ ~9,750 batches/epoch.
#   T_0 = COSINE_T0 epochs × steps_per_epoch for smooth cosine over 2 epochs.
steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0    = COSINE_T0 * steps_per_epoch,   # restart every T_0 epochs
    T_mult = 1,
    eta_min= LEARNING_RATE * 0.01,          # never drop below 1 % of base LR
)
print(f"   Total parameters  : {sum(p.numel() for p in model.parameters()):,}")
print(f"   Steps / epoch     : {steps_per_epoch:,}")
print(f"   Cosine T_0        : {COSINE_T0} epochs ({COSINE_T0 * steps_per_epoch:,} steps)")


# ════════════════════════════════════════════════════════════════════
# PHASE 1 — TRAINING
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 66)
print("  PHASE 1 — PRODUCTION TRAINING")
print("  Task      : Siamese ΔΔG regression (task='idr')")
print(f"  Loss      : AntisymmetricLoss (alpha={ALPHA_ASYM})")
print(f"  Scheduler : CosineAnnealingWarmRestarts (T_0={COSINE_T0} epochs)")
print(f"  Epochs    : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  Device : {device}")
print(f"  Dataset   : FULL MegaScale ({len(train_ds):,} train pairs)")
print("  Checkpointing: best + every epoch (preemption-safe)")
print("═" * 66)

history   = {"train_total":[], "train_fidelity":[], "train_antisymmetry":[],
             "val_total":[],   "val_fidelity":[],   "val_antisymmetry":  []}
best_val  = float("inf")
t_start   = time.time()


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss=tot_fid=tot_asym=n_b=0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False):
            try: wt, mt, lbl = make_siamese_pair(batch, device)
            except ValueError: continue
            if train: optimizer.zero_grad()
            dG_fwd, dG_rev = model((wt, mt), task="idr")
            loss, comps    = criterion(dG_fwd, dG_rev, lbl)
            if train:
                loss.backward(); optimizer.step(); scheduler.step()
            tot_loss+=loss.item(); tot_fid+=comps["fidelity"]; tot_asym+=comps["antisymmetry"]; n_b+=1
    n = max(n_b, 1)
    return tot_loss/n, tot_fid/n, tot_asym/n


for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_fid, tr_asym = run_epoch(train_loader, train=True)
    va_loss, va_fid, va_asym = run_epoch(val_loader,   train=False)
    elapsed = time.time() - t0

    history["train_total"].append(tr_loss); history["train_fidelity"].append(tr_fid)
    history["train_antisymmetry"].append(tr_asym)
    history["val_total"].append(va_loss);   history["val_fidelity"].append(va_fid)
    history["val_antisymmetry"].append(va_asym)

    # ── Epoch checkpoint (preemption-safe — always saved regardless of best) ──
    epoch_ckpt = CHECKPOINT_EPOCH.format(epoch=epoch)
    torch.save(model.state_dict(), epoch_ckpt)

    # ── Best checkpoint ───────────────────────────────────────────────────────
    best_flag = ""
    if va_loss < best_val:
        best_val = va_loss
        torch.save(model.state_dict(), CHECKPOINT_BEST)
        best_flag = "  ✅ best"

    print(
        f"Epoch {epoch:>2}/{EPOCHS}  "
        f"total={tr_loss:.4f}  fid={tr_fid:.4f}  asym={tr_asym:.4f}  "
        f"| val={va_loss:.4f}  val_fid={va_fid:.4f}  val_asym={va_asym:.4f}  "
        f"[{elapsed/60:.1f}min]  saved→{epoch_ckpt}{best_flag}"
    )

print(f"\n✅ Phase 1 complete in {(time.time()-t_start)/3600:.2f} h.")
print(f"   Best val loss : {best_val:.4f} → {CHECKPOINT_BEST}")


# ════════════════════════════════════════════════════════════════════
# PHASE 2 — BENCHMARKING
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 66)
print("  PHASE 2 — TEST-SET BENCHMARKING")
print("═" * 66)

model.load_state_dict(torch.load(CHECKPOINT_BEST, map_location=device))
model.eval()

all_preds, all_labels, asym_violations = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating test set"):
        try: wt, mt, lbl = make_siamese_pair(batch, device)
        except ValueError: continue
        dG_fwd, dG_rev = model((wt, mt), task="idr")
        fwd = dG_fwd.squeeze(-1).cpu(); rev = dG_rev.squeeze(-1).cpu(); lbl = lbl.cpu()
        all_preds.extend(fwd.tolist())
        all_labels.extend(lbl.tolist())
        asym_violations.extend((fwd + rev).abs().tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
asym_arr   = np.array(asym_violations)

rmse           = float(np.sqrt(np.mean((all_preds - all_labels)**2)))
mae            = float(np.mean(np.abs(all_preds - all_labels)))
pearson        = float(pearsonr(all_preds, all_labels)[0])
mean_asym_viol = float(asym_arr.mean())

print(f"\n  Test-set Benchmarks ({len(all_labels):,} pairs)")
print(f"  {'─'*45}")
print(f"  RMSE                   : {rmse:.4f} kcal/mol")
print(f"  MAE                    : {mae:.4f} kcal/mol")
print(f"  Pearson r              : {pearson:.4f}")
print(f"  Mean Antisymmetry Viol : {mean_asym_viol:.4f}  (target → 0.0)")


# ════════════════════════════════════════════════════════════════════
# PHASE 3 — VISUALISATION  → holognn_v5_metrics.png
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 66)
print("  PHASE 3 — VISUALISATION")
print("═" * 66)

plt.style.use("seaborn-v0_8-darkgrid")
ep_range = range(1, EPOCHS + 1)

fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    f"Holo-GNN V5.0 — Production Run (Full MegaScale, {len(full_ds):,} mutations)",
    fontsize=15, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.32)

# Plot 1 — Total AntisymmetricLoss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ep_range, history["train_total"], marker="o", label="Train")
ax1.plot(ep_range, history["val_total"],   marker="s", label="Val", linestyle="--")
ax1.set_title("Total AntisymmetricLoss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()

# Plot 2 — Fidelity Term
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ep_range, history["train_fidelity"], marker="o", color="steelblue",  label="Train")
ax2.plot(ep_range, history["val_fidelity"],   marker="s", color="dodgerblue", label="Val", linestyle="--")
ax2.set_title("Fidelity Term  (pred − exp)²"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("MSE")
ax2.legend()

# Plot 3 — Antisymmetry Term
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ep_range, history["train_antisymmetry"], marker="o", color="tomato",      label="Train")
ax3.plot(ep_range, history["val_antisymmetry"],   marker="s", color="lightsalmon", label="Val", linestyle="--")
ax3.set_title("Antisymmetry Term  (fwd + rev)²"); ax3.set_xlabel("Epoch"); ax3.set_ylabel("Penalty")
ax3.legend()

# Plot 4 — Predicted vs. Experimental ΔΔG scatter
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.scatter(all_labels, all_preds, alpha=0.25, s=4, color="mediumseagreen", label="Test pairs")
lims = [min(all_labels.min(), all_preds.min()), max(all_labels.max(), all_preds.max())]
ax4.plot(lims, lims, "k--", linewidth=1, label="y = x (ideal)")
ax4.set_xlabel("Experimental ΔΔG (kcal/mol)")
ax4.set_ylabel("Predicted ΔΔG (kcal/mol)")
ax4.set_title(
    f"Predicted vs. Experimental ΔΔG\n"
    f"RMSE={rmse:.4f}  MAE={mae:.4f}  Pearson r={pearson:.4f}"
)
ax4.legend(markerscale=4)

# Plot 5 — Antisymmetry violation histogram
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(asym_arr, bins=60, color="mediumpurple", edgecolor="white", linewidth=0.3)
ax5.axvline(mean_asym_viol, color="red", linestyle="--", label=f"Mean={mean_asym_viol:.4f}")
ax5.set_title("|dG_fwd + dG_rev| Distribution\n(Antisymmetry Violation)")
ax5.set_xlabel("|Violation| (kcal/mol)"); ax5.set_ylabel("Count")
ax5.legend()

plt.savefig("holognn_v5_metrics.png", dpi=150, bbox_inches="tight")
print("\n✅ Phase 3 complete — holognn_v5_metrics.png saved.")
plt.show()

print("\n" + "═" * 66)
print("  ALL PHASES COMPLETE — V5.0 PRODUCTION RUN")
print(f"  Best checkpoint : {CHECKPOINT_BEST}")
print(f"  Epoch checkpts  : holognn_v5_epoch_1.pth … holognn_v5_epoch_{EPOCHS}.pth")
print(f"  Metrics figure  : holognn_v5_metrics.png")
print(f"  RMSE            : {rmse:.4f} kcal/mol")
print(f"  Pearson r       : {pearson:.4f}")
print("═" * 66)